# Notebook 02: Preprocessing Pipeline & Empirical Ablation Study

**ComplaintScope: Consumer-Complaint Triage and Early Warning System**  
**Course:** CBS3009 – Advanced Social, Text and Media Analytics  
**Authors:** Ruhaan bansal (23BBS0046), Vedant Rajesh (23BBS0037), Anish Desai (23BBS0119)  
**Faculty:** Sathish Kumar K | **Institution:** VIT Vellore

### Objectives:
1. Demonstrate modular text preprocessing steps (Unicode normalization, noise stripping, CFPB redaction handling, tokenization, lemmatization, negation retention).
2. Empirically evaluate 7 distinct preprocessing configurations against validation Macro-F1 using TF-IDF + Logistic Regression.
3. Provide before-and-after narrative examples demonstrating redaction token preservation and vocabulary impact.
4. Document preprocessing rationale to support academic report and viva defense.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import load_config, get_path
from src.preprocess import (
    normalize_text, handle_redactions, remove_noise, lowercase, clean_tokens, preprocess_text
)

print('Setup complete. All modules imported successfully.')

## 1. Modular Preprocessing Architecture

The preprocessing pipeline executes in 6 systematic stages:
1. **Unicode Normalization (NFKC):** Harmonizes accents, ligatures, and formatting artifacts.
2. **Redaction Token Handling:** Maps CFPB mask patterns (`XXXX`, `XX/XX/XXXX`, `{$150.00}`) to structured tokens (`<redacted>`, `<date>`, `<amt>`) or deletes them.
3. **Noise Stripping:** Purges URLs, email addresses, phone numbers, and duplicate punctuations.
4. **Casing & Tokenization:** Lowercases text while retaining negation cues (`not`, `no`, `never`, `without`).
5. **Lemmatization:** Uses spaCy `en_core_web_sm` to reduce inflected words to lemmas while preserving part-of-speech context.
6. **Stopword Filtering:** Removes high-frequency uninformative terms while strictly preserving negation qualifiers.

In [ ]:
raw_example = 'I called XXXX on XX/XX/2021 regarding a {$150.25} fee that was charged to test@bank.com! Visit https://bank.com. The bank did NOT refund my money!'

print('Raw text:')
print(raw_example)
print('\n' + '-'*60)

s1 = normalize_text(raw_example)
s2 = handle_redactions(s1, mode='placeholder')
s3 = remove_noise(s2)
s4 = lowercase(s3)

print('Stage 1 (Normalized):', s1)
print('Stage 2 (Redactions Handled):', s2)
print('Stage 3 (Noise Removed):', s3)
print('Stage 4 (Lowercased):', s4)

full_cleaned = preprocess_text(raw_example, return_string=True)
print('\nFull Pipeline Output (Lemmatized + Negation Preserved):')
print(full_cleaned)

## 2. Empirical Preprocessing Ablation Study

To rigorously justify preprocessing decisions rather than relying on arbitrary defaults, we conduct an ablation study across 7 configurations on the validation split:
- **Baseline:** Full pipeline with redaction placeholders (`<amt>`, `<date>`, `<redacted>`), lemmatization, stopword removal, and negation preservation.
- **No Redaction Handling:** Deleting redaction markers (`XXXX`, `{$...}`) entirely.
- **No Lowercase:** Preserving original casing.
- **No Stopword Removal:** Retaining all stopwords.
- **No Lemmatization:** Using raw tokenization without morphological reduction.
- **No Negation Preservation:** Treating negation tokens as standard stopwords and deleting them.
- **Minimal:** Raw lowercasing without lemmatization, stopword filtering, or redaction token mapping.

In [ ]:
ablation_path = Path('../results/tables/ablation_preprocessing.csv') if Path('../results/tables/ablation_preprocessing.csv').exists() else Path('results/tables/ablation_preprocessing.csv')
df_ablation = pd.read_csv(ablation_path)
print('Ablation Study Results (Sorted by Macro-F1):')
display_cols = ['config', 'macro_f1', 'accuracy', 'elapsed_sec', 'lemmatize', 'remove_stopwords', 'keep_negations']
df_ablation[display_cols]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#10b981' if i == 0 else '#64748b' for i in range(len(df_ablation))]
bars = ax.barh(df_ablation['config'][::-1], df_ablation['macro_f1'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.8)

ax.set_xlabel('Validation Macro-F1 (TF-IDF + Logistic Regression)', fontsize=11, fontweight='bold')
ax.set_title('Figure: Preprocessing Ablation Study – Impact on Macro-F1', fontsize=12, fontweight='bold')
best_f1 = df_ablation['macro_f1'].max()
ax.axvline(x=best_f1, color='#f59e0b', linestyle='--', linewidth=1.5,
           label=f'Top Performing: {best_f1:.4f}')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.003, bar.get_y() + bar.get_height()/2,
            f'{width:.4f}', va='center', ha='left', fontsize=9, fontweight='semibold')

ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0.35, best_f1 + 0.06)
ax.grid(axis='x', alpha=0.3, linestyle=':')
plt.tight_layout()
plt.show()

## 3. Qualitative Before-and-After Narrative Transformation

Inspection of consumer narratives before and after preprocessing reveals the concrete effect on token length and semantic focus.

In [ ]:
examples_path = Path('../results/tables/ablation_text_examples.json') if Path('../results/tables/ablation_text_examples.json').exists() else Path('results/tables/ablation_text_examples.json')
with open(examples_path, 'r', encoding='utf-8') as f:
    ex = json.load(f)

print('=== RAW NARRATIVE SAMPLE (Excerpt) ===')
print(ex.get('raw_text', '')[:400] + '...')
print('\n=== PREPROCESSED NARRATIVE SAMPLE ===')
print(ex.get('processed_text', '')[:400] + '...')
print(f'\nSample sizes used in ablation: Train={ex.get("ablation_sample_size_train")}, Val={ex.get("ablation_sample_size_val")}')

## 4. Empirical Takeaways & Pipeline Selection

1. **Impact of Lemmatization:** Omitting lemmatization causes a major decline in Macro-F1 (dropping from ~0.50-0.51 to 0.4635), demonstrating that morphological normalization is essential for grouping disparate complaints.
2. **Impact of Stopwords:** Omitting stopword removal causes performance to collapse to 0.4236 Macro-F1, as high-frequency boilerplate tokens drown out issue-specific terminology.
3. **Negation Preservation:** Preserving negation markers (`not`, `no`, `never`) prevents critical reversal of sentiment in consumer dispute contexts.
4. **Redaction Treatment:** Both placeholder mapping and redaction deletion perform strongly; stripping redaction strings provides clean text representation for downstream modeling.